# Epic 4 -- Domain-Adversarial Neural Network (DANN) Overview

This notebook introduces the DANN approach used in Epic 4 for **multi-site void
segmentation**.  The core problem: a segmentation model trained on X-ray images from
one manufacturing site (the *source* domain) often degrades when applied to images
from a different site (the *target* domain) due to differences in equipment,
acquisition settings, and part geometry.

DANN solves this with **domain-adversarial training** -- the encoder learns features
that are useful for segmentation but *indistinguishable* between domains.

## What you will learn

1. The DANN architecture: shared encoder + segmentation decoder + domain classifier
2. How the Gradient Reversal Layer (GRL) works
3. How to create a `DANNModel` and run a forward pass
4. The lambda schedule that stabilises early training

## Prerequisites

- Python 3.12, PyTorch, timm
- Install: `uv pip install -e ".[epic4]"`

---
## 1. Architecture Overview

The DANN model has three components:

```
                         +-------------------+
                         |  Input Image      |
                         |  [B, 3, 512, 512] |
                         +--------+----------+
                                  |
                                  v
                    +----------------------------+
                    |     Shared Encoder          |
                    |  (ConvNeXt-Tiny backbone)   |
                    |  4 multi-scale feature maps  |
                    +---+--------------------+----+
                        |                    |
                        | skip connections   | bottleneck features
                        v                    v
              +------------------+   +-----------------------+
              |  U-Net Decoder   |   | Gradient Reversal     |
              |  (segmentation)  |   | Layer (GRL)           |
              +--------+---------+   +----------+------------+
                       |                        |
                       v                        v
              +------------------+   +-----------------------+
              | Seg Logits       |   | Domain Classifier     |
              | [B, 1, H, W]    |   | source=0 / target=1   |
              +------------------+   | [B, 1]                |
                                     +-----------------------+
```

### Key idea

| Component | Purpose |
|-----------|--------|
| **Shared Encoder** | Extract features from input images (any timm backbone) |
| **U-Net Decoder** | Upsample features back to input resolution for pixel-level segmentation |
| **GRL + Domain Classifier** | Force the encoder to produce domain-invariant features by *reversing* gradients |

The GRL is the magic: during **forward** pass it does nothing (identity), but during
**backward** pass it *negates* the gradients.  This means the encoder is trained to
*maximally confuse* the domain classifier -- i.e., learn features where source and
target look the same.

---
## 2. Quick Code Demo: Creating and Running a DANNModel

Let's instantiate the model and run a forward pass on random data.

In [ ]:
import torch
from udm_epic4.models.dann import DANNModel
from udm_epic4.models.encoder import SharedEncoder
from udm_epic4.models.decoder import UNetDecoder
from udm_epic4.models.domain_classifier import DomainClassifier, GradientReversalLayer

In [ ]:
# Create a DANN model with the default ConvNeXt-Tiny backbone.
# Set pretrained=False for a quick demo (no download needed).
model = DANNModel(
    backbone="convnext_tiny",
    pretrained=False,          # set True for real training
    in_chans=3,
    decoder_channels=[256, 128, 64, 32],
    domain_head_hidden=256,
)
print(f"Model created.  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Simulate a batch of 4 images, 3 channels, 512x512
dummy_input = torch.randn(4, 3, 512, 512)

# Forward pass with lambda=0.5 (mid-training)
model.eval()
with torch.no_grad():
    seg_logits, domain_logits = model(dummy_input, lambda_val=0.5)

print(f"Segmentation logits shape: {seg_logits.shape}")    # [4, 1, 512, 512]
print(f"Domain logits shape:       {domain_logits.shape}")  # [4, 1]

**What the outputs mean:**

- `seg_logits [B, 1, H, W]`: raw (pre-sigmoid) segmentation predictions.  Apply
  `torch.sigmoid(seg_logits) > 0.5` to get binary masks.
- `domain_logits [B, 1]`: raw domain classification scores.  Apply `torch.sigmoid`
  to get probability of being the *target* domain (label=1).

In [ ]:
# You can also extract just the bottleneck features (useful for t-SNE analysis).
with torch.no_grad():
    bottleneck = model.encode(dummy_input)

print(f"Bottleneck features shape: {bottleneck.shape}")  # [4, 768, 16, 16] for ConvNeXt-Tiny

### Inspecting the encoder

The `SharedEncoder` wraps any timm backbone with `features_only=True`, producing
4 feature maps at decreasing spatial resolution:

In [ ]:
print("Encoder feature channels (shallow -> deep):")
print(model.encoder.feature_channels)  # e.g. [96, 192, 384, 768]

---
## 3. Lambda Schedule

The GRL strength (`lambda_val`) is **not** constant during training.  If we set it
to 1.0 from the start, the domain adversarial signal would destabilise the encoder
before it has learned useful features.

Instead, we use the schedule from Ganin et al. (2016):

$$\lambda(p) = \lambda_{\max} \cdot \frac{2}{1 + \exp(-10p)} - 1$$

where $p = \text{current\_epoch} / \text{max\_epochs}$ (progress from 0 to 1).

This starts near 0 and ramps up in a sigmoid curve to $\lambda_{\max}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from udm_epic4.training.lambda_scheduler import dann_lambda_schedule

# Compute lambda for every progress value from 0 to 1
progress = np.linspace(0.0, 1.0, 200)
lambdas = [dann_lambda_schedule(p, lambda_max=1.0) for p in progress]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(progress, lambdas, linewidth=2, color="#2196F3")
ax.set_xlabel("Training progress (epoch / max_epochs)", fontsize=12)
ax.set_ylabel("Lambda (GRL strength)", fontsize=12)
ax.set_title("DANN Lambda Schedule", fontsize=14)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="lambda=0.5")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

**Reading the plot:**

- **Early training (progress ~ 0):** Lambda is near 0, so the GRL has almost no
  effect. The encoder trains mainly on segmentation loss.
- **Mid training (progress ~ 0.5):** Lambda is around 0.75, the domain adversarial
  signal starts pushing the encoder toward domain-invariant features.
- **Late training (progress ~ 1.0):** Lambda saturates near 1.0, the full
  adversarial signal is applied.

You can control the ceiling with `lambda_max` in `configs/epic4_dann.yaml`.

---
## 4. How the Gradient Reversal Layer Works

The GRL is implemented as a custom `torch.autograd.Function`:

```python
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.clone()            # identity in forward

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_val * grad_output, None  # negate + scale in backward
```

This means:
- The domain classifier receives the *same* features as the decoder (no distortion).
- But the encoder sees *negated* gradients from the domain loss.
- So the encoder is simultaneously optimised to:
  1. **Minimise** segmentation loss (via the decoder path)
  2. **Maximise** domain classification loss (via the GRL path)

The result is a minimax game: the domain classifier tries to distinguish domains,
while the encoder tries to fool it.

---
## 5. Project Structure

| Module | Description |
|--------|------------|
| `udm_epic4.models.encoder` | `SharedEncoder` -- timm backbone wrapper |
| `udm_epic4.models.decoder` | `UNetDecoder` -- skip-connection segmentation head |
| `udm_epic4.models.domain_classifier` | `GradientReversalLayer`, `DomainClassifier` |
| `udm_epic4.models.dann` | `DANNModel` -- composes all three |
| `udm_epic4.data.multi_domain_dataset` | `DomainDataset`, `build_datasets_from_config` |
| `udm_epic4.data.domain_sampler` | `DomainBatchSampler` -- balanced source/target batches |
| `udm_epic4.training.lambda_scheduler` | `dann_lambda_schedule` |
| `udm_epic4.training.train_baseline` | Source-only training (US 4.2) |
| `udm_epic4.training.train_dann` | DANN adversarial training (US 4.3) |
| `udm_epic4.evaluation.metrics` | `compute_f1`, `compute_iou`, `compute_dice`, `evaluate_all_domains` |
| `udm_epic4.evaluation.domain_analysis` | `extract_features`, `compute_tsne`, `domain_confusion_score` |
| `udm_epic4.reporting.failure_analysis` | `categorize_failures`, `generate_failure_report` |

## Next steps

| Notebook | Topic |
|----------|------|
| `epic4_01_data_analysis.ipynb` | Load and visualise multi-site data |
| `epic4_02_baseline.ipynb` | Train a source-only model (no adaptation) |
| `epic4_03_dann_training.ipynb` | Train with DANN + GRL |
| `epic4_04_domain_analysis.ipynb` | t-SNE, domain confusion analysis |
| `epic4_05_deployment.ipynb` | Evaluate on all sites |
| `epic4_06_failure_analysis.ipynb` | Failure mode report |